### 生成 Node2Vec 嵌入

In [5]:
import torch
from torch_geometric.nn import Node2Vec
import pandas as pd
import numpy as np
import pickle
import os
import gc  # 引入垃圾回收模块
import warnings

# 忽略警告
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

# ================= 配置 =================
DATA_PATH = '../data/kg.csv'
OUTPUT_NPY = '../data/node2vec_embeddings.npy'
OUTPUT_MAP = '../data/node2vec_id_map.pkl'

EMBEDDING_DIM = 128
WALK_LENGTH = 30
CONTEXT_SIZE = 10
WALKS_PER_NODE = 10
EPOCHS = 50
BATCH_SIZE = 256

# =====================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f">> 使用设备: {device}")

# 1. 加载数据
print(">> 正在加载数据...")
df = pd.read_csv(DATA_PATH, dtype=str, low_memory=False)
print(f"✅ 数据加载完成，共 {len(df)} 条边。")

# 2. 构建图索引
node_to_id = {}
id_to_node = []
edge_index_list = [[], []]

def get_id(node_str):
    if node_str not in node_to_id:
        node_to_id[node_str] = len(node_to_id)
        id_to_node.append(node_str)
    return node_to_id[node_str]

print(">> 正在构建图索引 (这可能有点慢，请耐心等待)...")
# 简单的迭代，减少开销
for _, row in df.iterrows():
    src = f"{row['x_type']}::{row['x_index']}"
    dst = f"{row['y_type']}::{row['y_index']}"
    
    u = get_id(src)
    v = get_id(dst)
    
    edge_index_list[0].append(u)
    edge_index_list[1].append(v)

edge_index = torch.tensor(edge_index_list, dtype=torch.long)
num_nodes = len(node_to_id)

print(f"\n✅ 图构建完成！节点数: {num_nodes}, 边数: {edge_index.shape[1]}")

# 🚀【关键优化】手动释放 DataFrame 和列表的内存，为模型训练腾出空间
del df
del edge_index_list
gc.collect() 
print(">> 已清理原始数据内存，准备初始化模型...")

# 3. 初始化模型
model = Node2Vec(
    edge_index=edge_index,
    embedding_dim=EMBEDDING_DIM,
    walk_length=WALK_LENGTH,
    context_size=CONTEXT_SIZE,
    walks_per_node=WALKS_PER_NODE,
    sparse=True
).to(device)

# 释放 edge_index (模型内部已经持有了引用，或者我们可以保留它，视 PyG 版本而定)
# 为了安全，我们保留 edge_index 直到 loader 创建完成
loader = model.loader(batch_size=BATCH_SIZE)
del edge_index  # loader 创建后可以删除原始的 edge_index 张量
gc.collect()

optimizer = torch.optim.SparseAdam(list(model.parameters()), lr=0.01)

# 4. 训练
print(">> 开始训练 Node2Vec...")
model.train()

for epoch in range(1, EPOCHS + 1):
    total_loss = 0
    # 注意：如果 NUM_WORKERS > 0，loader 是多进程的，这里不会阻塞太多内存
    for pos_rw, neg_rw in loader:
        optimizer.zero_grad()
        loss = model.loss(pos_rw.to(device), neg_rw.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss / len(loader):.4f}")

print("✅ 训练完成！正在保存结果...")

# 5. 提取并保存 (优化版：拆分为 .npy 和 .pkl)
model.eval()
with torch.no_grad():
    # 获取 CPU 上的 numpy 数组
    embedding_matrix = model.embedding.weight.cpu().numpy()

# 保存映射表 (只存字符串列表，很小)
with open(OUTPUT_MAP, 'wb') as f:
    pickle.dump(id_to_node, f, protocol=pickle.HIGHEST_PROTOCOL)

# 保存矩阵 (二进制，极快且小)
np.save(OUTPUT_NPY, embedding_matrix)

# 计算文件大小
size_npy = os.path.getsize(OUTPUT_NPY) / 1024 / 1024
size_map = os.path.getsize(OUTPUT_MAP) / 1024 / 1024
print(f"\n✅ 成功保存！")
print(f"   矩阵文件 (.npy): {size_npy:.2f} MB")
print(f"   映射文件 (.pkl): {size_map:.2f} MB")
print(f"   总计: {size_npy + size_map:.2f} MB (远小于之前的 8GB+)")

# 🚀【关键优化】彻底清理模型，防止后续代码爆内存
del model
del embedding_matrix
del id_to_node
del node_to_id
gc.collect()

print(">> 内存已清理，脚本安全结束。你可以安全地加载 .npy 和 .pkl 文件进行下一步了。")

>> 使用设备: cuda
>> 正在加载数据...
✅ 数据加载完成，共 8100498 条边。
>> 正在构建图索引 (这可能有点慢，请耐心等待)...

✅ 图构建完成！节点数: 129375, 边数: 8100498
>> 已清理原始数据内存，准备初始化模型...
>> 开始训练 Node2Vec...
Epoch 10, Loss: 0.8487
Epoch 20, Loss: 0.8405
Epoch 30, Loss: 0.8396
Epoch 40, Loss: 0.8392
Epoch 50, Loss: 0.8392
✅ 训练完成！正在保存结果...

✅ 成功保存！
   矩阵文件 (.npy): 63.17 MB
   映射文件 (.pkl): 2.82 MB
   总计: 65.99 MB (远小于之前的 8GB+)
>> 内存已清理，脚本安全结束。你可以安全地加载 .npy 和 .pkl 文件进行下一步了。


###生成 PubMedBERT 嵌入

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

print(">> 正在生成 PubMedBERT 嵌入...")
# df = pd.read_csv('../data/kg.csv')
tokenizer = AutoTokenizer.from_pretrained("microsoft/pubmedbert-base-uncased")
model = AutoModel.from_pretrained("microsoft/pubmedbert-base-uncased").cuda()
model.eval()

# 收集所有唯一节点及其名称
nodes_map = {} # id_str -> name_str
for _, row in df.iterrows():
    # 源节点
    src_id = f"{row['x_type']}::{row['x_index']}"
    src_name = row['x_name'] if pd.notna(row['x_name']) else f"{row['x_type']} {row['x_index']}"
    if src_id not in nodes_map: nodes_map[src_id] = src_name
    
    # 目标节点 
    dst_id = f"{row['y_type']}::{row['y_index']}"
    dst_name = row['y_name'] if pd.notna(row['y_name']) else f"{row['y_type']} {row['y_index']}"
    if dst_id not in nodes_map: nodes_map[dst_id] = dst_name

unique_ids = list(nodes_map.keys())
unique_texts = [nodes_map[i] for i in unique_ids]

embeddings = []
batch_size = 32
with torch.no_grad():
    for i in tqdm(range(0, len(unique_texts), batch_size)):
        batch_texts = unique_texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).cuda()
        outputs = model(**inputs)
        embeddings.append(outputs.last_hidden_state[:, 0, :].cpu()) # CLS token

final_emb = torch.cat(embeddings, dim=0)
records = [{'id': uid, 'embedding': final_emb[i]} for i, uid in enumerate(unique_ids)]
pd.DataFrame(records).to_pickle('../data/pubmedbert_embeddings.pkl')
print("✅ PubMedBERT 完成: ../data/pubmedbert_embeddings.pkl")